In [4]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd

In [5]:



try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
modality="visual"
layer_script = "event"
subj= "s01b"


# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, modality=modality,layer_script=layer_script,  subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_eve

In [6]:
#epochs

subjects_BV = []

# Busca archivos .vhdr dentro de export_generic_data
for archivo in export_generic_data.glob("*.vhdr"):
    nombre = archivo.stem  # sin la extensión .vhdr

    # Ejemplo: s01b_vis_c_BV_mne -> queremos s01b
    sujeto = nombre.split("_")[0]

    subjects_BV.append(sujeto)


subjects_BV = sorted(set(subjects_BV), key=str.lower)
print(subjects_BV)


['s01b', 's02b', 's03b', 's04b', 's05b', 's06b', 's07b', 's08b', 's09b', 's10b', 's11b', 's12b', 's13b', 's14b', 's15b', 's16b', 's17b', 's18b', 'S19b', 'S20b', 'S21b', 'S22b', 'S23b', 'S24b', 'S25b', 'S26b', 'S27b', 'S28b', 'S29b', 'S30b', 'S31b', 's32b', 'S33b', 'S34b', 'S35b', 's36b']


# CANALES VISUALES

In [8]:
from collections import defaultdict
import mne

# Dictionaries to store results
channels_by_subject = {}
subjects_missing_channels = {}
subjects_extra_channels = {}

# Reference variables
reference_subject = None
reference_channels_by_type = None

for subj in subjects_BV:

    # --- Find vhdr file flexibly ---
    files = list(export_generic_data.glob(f"{subj}*.vhdr"))

    if len(files) == 0:
        print(f"❌ No file found for {subj}")
        continue

    elif len(files) > 1:
        print(f"⚠️ Multiple files found for {subj}:")
        for f in files:
            print(f)
        continue

    file_path = files[0]
    print(f"📂 Loading: {file_path.name}")

    try:
        # Load BrainVision file without preloading data
        raw = mne.io.read_raw_brainvision(file_path, preload=False)

        # Set EOG channel types
        raw.set_channel_types({
            "HEOG+": "eog",
            "HEOG": "eog",
            "VEOG+": "eog",
            "VEOG": "eog",
        })

        # Group channels by type
        channels_by_type = defaultdict(list)
        for ch_name, ch_type in zip(raw.ch_names, raw.get_channel_types()):
            channels_by_type[ch_type].append(ch_name)

        # Sort channel names within each type
        for ch_type in channels_by_type:
            channels_by_type[ch_type] = sorted(channels_by_type[ch_type])

        channels_by_subject[subj] = channels_by_type

        # Use first valid subject as reference
        if reference_subject is None:
            reference_subject = subj
            reference_channels_by_type = {
                ch_type: set(ch_list)
                for ch_type, ch_list in channels_by_type.items()
            }
            print(f"✅ Reference subject: {reference_subject}")

        print(f"✅ Raw loaded correctly for {subj}")

    except Exception as e:
        print(f"❌ Error loading raw for {subj}: {e}")
        continue


# --- Print reference channel list ---
print(f"\n📌 Reference subject: {reference_subject}")
print("📌 Reference channels by type:")

for ch_type in sorted(reference_channels_by_type.keys()):
    print(f"\n{ch_type.upper()} ({len(reference_channels_by_type[ch_type])}):")
    print(sorted(reference_channels_by_type[ch_type]))


# --- Compare all subjects against reference ---
for subj, subj_channels_by_type in channels_by_subject.items():
    if subj == reference_subject:
        continue

    subj_missing = {}
    subj_extra = {}

    # Compare all channel types present in either subject
    all_types = set(reference_channels_by_type.keys()).union(set(subj_channels_by_type.keys()))

    for ch_type in sorted(all_types):
        ref_channels = reference_channels_by_type.get(ch_type, set())
        subj_channels = set(subj_channels_by_type.get(ch_type, []))

        missing_channels = sorted(ref_channels - subj_channels)
        extra_channels = sorted(subj_channels - ref_channels)

        if missing_channels:
            subj_missing[ch_type] = missing_channels

        if extra_channels:
            subj_extra[ch_type] = extra_channels

    if subj_missing:
        subjects_missing_channels[subj] = subj_missing

    if subj_extra:
        subjects_extra_channels[subj] = subj_extra


# --- Print summary of missing channels ---
print("\n📋 Subjects with missing channels:")
if subjects_missing_channels:
    for subj, missing_info in subjects_missing_channels.items():
        print(f"\n{subj}")
        for ch_type, ch_list in missing_info.items():
            print(f"  ❌ Missing {ch_type}: {ch_list}")
else:
    print("✅ No subjects with missing channels")


# --- Print summary of extra channels ---
print("\n📋 Subjects with extra channels:")
if subjects_extra_channels:
    for subj, extra_info in subjects_extra_channels.items():
        print(f"\n{subj}")
        for ch_type, ch_list in extra_info.items():
            print(f"  ✅ Extra {ch_type}: {ch_list}")
else:
    print("✅ No subjects with extra channels")

📂 Loading: s01b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s01b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Reference subject: s01b
✅ Raw loaded correctly for s01b
📂 Loading: s02b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s02b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for s02b
📂 Loading: s03b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s03b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for s03b


C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: No coordinate information found for channels ['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: Not setting positions of 6 misc channels found in montage:
['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:36: RuntimeWarning: The unit for channel(s) HEOG, HEOG+, VEOG, VEOG+ has changed from NA to V.
  raw.set_channel_types({
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: Run

📂 Loading: s04b_vis_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s04b_vis_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for s04b
📂 Loading: s05b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s05b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for s05b
📂 Loading: s06b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s06b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for s06b


C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: No coordinate information found for channels ['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: Not setting positions of 6 misc channels found in montage:
['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:36: RuntimeWarning: The unit for channel(s) HEOG, HEOG+, VEOG, VEOG+ has changed from NA to V.
  raw.set_channel_types({
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: Run

📂 Loading: s07b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s07b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for s07b
📂 Loading: s08b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s08b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for s08b
📂 Loading: s09b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s09b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for s09b
📂 Loading: s10b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s10b_vis_c_BV_mne.vhdr...


...

C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:36: RuntimeWarning: The unit for channel(s) HEOG, HEOG+, VEOG, VEOG+ has changed from NA to V.
  raw.set_channel_types({
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: No coordinate information found for channels ['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: Not setting positions of 6 misc channels found in montage:
['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:36: Run

Setting channel info structure...
✅ Raw loaded correctly for s32b
📂 Loading: S33b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\S33b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for S33b
📂 Loading: S34b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\S34b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for S34b
📂 Loading: S35b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\S35b_vis_c_BV_mne.vhdr...
Setting channel info structure...


C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: No coordinate information found for channels ['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: Not setting positions of 6 misc channels found in montage:
['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:36: RuntimeWarning: The unit for channel(s) HEOG, HEOG+, VEOG, VEOG+ has changed from NA to V.
  raw.set_channel_types({
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: Run

✅ Raw loaded correctly for S35b
📂 Loading: s36b_vis_c_BV_mne.vhdr
Extracting parameters from g:\PROYECTO_SELF\SELF_visual\BRAIN_VISION_SELF\export_generic_data\s36b_vis_c_BV_mne.vhdr...
Setting channel info structure...
✅ Raw loaded correctly for s36b

📌 Reference subject: s01b
📌 Reference channels by type:

EEG (59):
['AF3', 'AF4', 'AF7', 'AF8', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'CP1', 'CP2', 'CP3', 'CP4', 'CP5', 'CP6', 'CPz', 'Cz', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'FC1', 'FC2', 'FC3', 'FC4', 'FC5', 'FC6', 'FCz', 'FT7', 'FT8', 'Fp1', 'Fp2', 'Fpz', 'Fz', 'O1', 'O2', 'Oz', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'PO3', 'PO4', 'PO7', 'PO8', 'Pz', 'T7', 'T8', 'TP7', 'TP8']

EOG (4):
['HEOG', 'HEOG+', 'VEOG', 'VEOG+']

MISC (2):
['M1', 'M2']

📋 Subjects with missing channels:
✅ No subjects with missing channels

📋 Subjects with extra channels:
✅ No subjects with extra channels


C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: Not setting positions of 6 misc channels found in montage:
['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:36: RuntimeWarning: The unit for channel(s) HEOG, HEOG+, VEOG, VEOG+ has changed from NA to V.
  raw.set_channel_types({
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: RuntimeWarning: No coordinate information found for channels ['HEOG+', 'HEOG', 'VEOG+', 'VEOG', 'M1', 'M2']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(file_path, preload=False)
C:\Users\UCM\AppData\Local\Temp\ipykernel_2096\3832868428.py:33: Run

In [12]:

# Use reference subject (already loaded before)
reference_channel_list = raw.ch_names
reference_channel_types = raw.get_channel_types()

# Create dataframe preserving original order
channels_df = pd.DataFrame({
    "channel": reference_channel_list,
    "type": reference_channel_types
})

# Save CSV
channels_df.to_csv(
    channels_structure_path / f"channels_{modality}.csv",
    index=False
)

print("✅ Channel CSV saved with original order")

✅ Channel CSV saved with original order


In [13]:
channels_structure_path

WindowsPath('g:/PROYECTO_SELF/SELF_visual/channels_structure')

In [14]:
channels_df

,channel,type
0,Fp1,eeg
1,Fpz,eeg
2,Fp2,eeg
3,AF7,eeg
4,AF3,eeg
...,...,...
60,HEOG,eog
61,VEOG+,eog
62,VEOG,eog
63,M1,misc
